In [1]:
from rich.traceback import install
from omegaconf import OmegaConf
import wandb
import torch
from PIL import Image

In [2]:
# Import autoencoder utility tools
from mnist_autoencoders.data.utils import train_loader, test_loader, output_to_image
from mnist_autoencoders.models.VAE import VAE
from mnist_autoencoders.utils import vae_inference_utils

In [3]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cuda"

In [4]:
install(show_locals=True)
cfg = OmegaConf.load("../configs/vae.yaml")
wandb.init(
    project="mnist-autoencoders",
    name=cfg.run_name,
    config=OmegaConf.to_container(cfg, resolve=True),
)
vae = VAE(cfg).to(device)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jess-price/.netrc.
wandb: Currently logged in as: jessprice144 (jessprice144-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# Now i need to define the model and an optimiser
optimiser = torch.optim.Adam(
    vae.parameters(), lr=cfg.training.lr, betas=(0.9, 0.999), eps=1e-8
)

In [6]:
# Now i need to define the training loop and, inside, the loss function
vae_inference_utils = vae_inference_utils(vae, cfg, device, wandb, optimiser=optimiser)
for epoch in range(cfg.training.epochs):
    vae_inference_utils.train_test_loop(
        mode="train", epoch=epoch, data_loader=train_loader
    )
    with torch.inference_mode():
        vae_inference_utils.train_test_loop(
            mode="test", epoch=epoch, data_loader=test_loader
        )
wandb.finish()

epoch: 0:   0%|                                                        | 0/938 [00:00<?, ?it/s]/home/jess-price/Documents/mnist-autoencoders/src/mnist_autoencoders/utils.py:50: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  latent_log_nrml = F.log_softmax(latent_nrml)
epoch: 49: 100%|████████████████████████████████████████████| 938/938 [00:02<00:00, 322.48it/s]
                                                                                               

test/construction_loss,█▆▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test/kl_loss,█▂▁▁▂▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test/total_loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/construction_loss,█▅▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/kl_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total_loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test/construction_loss,0.16402
test/kl_loss,0.00134
test/total_loss,0.16416
train/construction_loss,0.16602
train/kl_loss,0.00177


# Results

In [7]:
# Here I shall test the model visually, passing an image through,
# denormalising the result and rendering it with pillow
data = next(iter(train_loader))[0][1].to(device)

output = vae(torch.reshape(data, (1, 1, 28, 28)))[0][0]
restored_output = output_to_image(output)
img_hat = Image.fromarray(restored_output.cpu().detach().numpy())

restored_data = output_to_image(data)[0]
img = Image.fromarray(restored_data.cpu().numpy())

In [8]:
img

In [9]:
img_hat

In [10]:
torch.save(vae.state_dict(), "../models/vae_v1/checkpoint_epoch_50.pt")